# Static GNN end-to-end overhead timing — root and B&B

**Purpose:** Re-use the saved final models, rebuild the *same* tripartite test graphs, and time per-instance graph construction, train-fitted feature normalization, one-graph PyG batching/device transfer, and forward inference. **No training, solver calls, or new predictions are used for the selection results.** Existing `preds_test.csv` decisions are verified against the reloaded weights (mismatches cause failure). 

**Critical:** The two original runner scripts saved `model.pt` but not train-fitted `FeatureNorm.stats`. Normalizers must be reconstructed from original training graphs. This expensive reconstruction occurs **outside the measured test-instance overhead** and uses train data only, but reflects one-time offline preparation. Times are for **single-instance online inference**, not the training pipeline's `BATCH_EVAL=16` batched evaluation. The original run uses CPU; benchmark on CPU for consistency. Do not time PKL file loading or training-set normalization fitting as per-instance cost.

Place `root_gnn_original.py` and `bb_gnn_original.py` **in the same folder as this notebook**. The original scripts' Windows `ROOT` paths must exist. Expected run directories are fixed, with **B&B October 21**, root October 24. Default no dynamic points.

In [1]:
from pathlib import Path
import runpy, time, numpy as np, pandas as pd, torch
from torch_geometric.data import Batch

FINAL=Path(r'C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib')
ROOT_RUN=FINAL/'gnn_ROOT_8_designs_UNBALANCED_20261024_165255_822932'
BB_RUN=FINAL/'gnn_synprime_8016_20261021_020453_440418'
OUT=FINAL/'analysis3_gap_time_outputs_WITH_GNN'
OUT.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cpu') # original completed on CPU
WARMUPS=2
REPEATS=3
assert ROOT_RUN.is_dir() and BB_RUN.is_dir(), 'Saved model run folder missing'
root=runpy.run_path(str(Path('root_gnn_original.py').resolve()),run_name='gnn_timing_root_import')
bb=runpy.run_path(str(Path('bb_gnn_original.py').resolve()),run_name='gnn_timing_bb_import')
print('Imported original GNN definitions without executing main/training.')

Imported original GNN definitions without executing main/training.


## Prepare the original training normalizer and test instances

Only Synthetic′→Combined models are used in Analysis 3. The root graph normalizer is refitted from the exact 6,068 Synthetic′ training instances, while the B&B normalizer uses its 6,069 Synthetic′ training instances. Build the normalizers separately because their training populations differ. The graph construction uses the original script functions verbatim.

In [2]:
def prepare(kind, m):
    if kind=='root':
        train=m['load_root_target']('synthetic','LP_vs_SDP_PRIME')
        test=m['load_root_target']('combined','LP_vs_SDP_PRIME')
    else:
        train,_=m['load_targets']('synthetic')
        test,_=m['load_targets']('combined')
    train=train.copy();test=test.copy()
    train['_dataset_kind']='synthetic';test['_dataset_kind']='combined'
    ids_train=set(train['_key']);ids_test=set(test['_key'])
    insts={}
    insts.update(m['load_combined_instances'](ids_test))
    insts.update(m['load_synthetic_instances'](ids_train))
    # The runners themselves create a LIST of all training graphs to fit norm.
    # This is deliberately untimed: one-time offline normalization calibration.
    raw_ds=m['QCQPDataset'](insts,train,None,kind+'_train_raw')
    norm=m['FeatureNorm']()
    print(kind,'refitting train-only graph feature statistics:',len(train),'graphs')
    norm.fit([raw_ds.build_raw(i) for i in range(len(raw_ds))])
    return test,insts,norm

# This preparation can be substantial in CPU/RAM, but runs only once per kind.
prepared={kind:prepare(kind,mod) for kind,mod in [('root',root),('bb',bb)]}
print('Prepared both normalizers; no models were trained.')

[01:27:04] synthetic LP_vs_SDP_PRIME: 6068 rows, labels {0: 3934, 1: 2134}
[01:27:04] combined LP_vs_SDP_PRIME: 278 rows, labels {0: 98, 1: 180}
[01:27:05] [PKL] MINLPLIB | qcp_lpall: matched 138 eligible target instances
[01:27:06] [PKL] MINLPLIB | qcp_lpopf: matched 4 eligible target instances
[01:27:06] [PKL] MINLPLIB | qcp_lpbigpkl: matched 4 eligible target instances
[01:27:21] [PKL] QPLIB | pickles_nlt300_mge1_bounded: matched 132 eligible target instances
[01:27:27] Synthetic' graph IDs matched via instance_id: 6068
[01:27:27] [root_train_raw] dataset size=6068
root refitting train-only graph feature statistics: 6068 graphs
[01:28:51] [combined probe10] zero-LP-root exclusion: removed 3/283; kept 280
[01:28:52] [PKL] MINLPLIB | qcp_lpall: matched 139 eligible target instances
[01:28:53] [PKL] MINLPLIB | qcp_lpopf: matched 4 eligible target instances
[01:28:54] [PKL] MINLPLIB | qcp_lpbigpkl: matched 4 eligible target instances
[01:29:30] [PKL] QPLIB | pickles_nlt300_mge1_bounded:

## Time both saved heads, with prediction consistency checks

`graph_time_s` starts at the in-memory QCQP instance and ends after `build_graph`. `normalize_time_s` includes application of the frozen, train-fitted statistics. `inference_time_s` includes forming a one-graph `Batch`, moving to CPU, one forward pass, and conversion of the selected decision. Measurements use repeated single-instance calls. Root/GNN-C uses `run_07`; root/GNN-R uses `run_08`; B&B/GNN-C uses `run_03`; B&B/GNN-R uses `run_04`.

No test labels are used as model inputs: the builders attach labels for the training interface, but the graph forward pass accesses only graph attributes. Existing prediction outputs are the source of truth for Analysis 3, so the code aborts if recomputed decisions disagree.

In [3]:
def run_directory(kind,task):
    parent=ROOT_RUN if kind=='root' else BB_RUN
    prefix=('run_07__' if task=='classification' else 'run_08__') if kind=='root' else ('run_03__' if task=='classification' else 'run_04__')
    hits=[p for p in parent.glob(prefix+'*') if p.is_dir()]
    if len(hits)!=1:raise RuntimeError(f'{kind} {task}: expected 1 run, got {hits}')
    return hits[0]

def predict_one(model,graph,task):
    batch=Batch.from_data_list([graph]).to(DEVICE)
    with torch.inference_mode():
        direct,_=model(batch)
        value=float(direct.item())
        if task=='classification':return 'LP' if value>=0 else 'SDP_PRIME'
        return 'LP' if value>-0.01 else 'SDP_PRIME'

def benchmark(kind, task, m, test, instances, norm):
    rd=run_directory(kind,task)
    ckpt=rd/'model.pt'; predfile=rd/'preds_test.csv'
    assert ckpt.is_file() and predfile.is_file(),f'Missing checkpoint/preds in {rd}'
    expected=pd.read_csv(predfile)
    assert expected['_key'].is_unique
    expected=expected.set_index('_key')['selected_relaxation'].astype(str).str.upper().str.replace("'",'_PRIME',regex=False).str.replace('SDP_PRIME','SDP_PRIME',regex=False)
    expected=expected.str.replace('SDP_PRIME','SDP_PRIME',regex=False)
    assert set(test['_key'])==set(expected.index), f'{kind} {task}: test IDs do not equal saved prediction IDs'
    model=m['TripartiteGNN'](m['HIDDEN'],m['LAYERS'],m['DROPOUT'],two_heads=(task=='classification')).to(DEVICE)
    try:weights=torch.load(ckpt,map_location=DEVICE,weights_only=True)
    except TypeError:weights=torch.load(ckpt,map_location=DEVICE)
    model.load_state_dict(weights,strict=True)
    model.eval()
    results=[]
    for idx,row in test.reset_index(drop=True).iterrows():
        key=str(row['_key']);inst=instances[('combined',key)]
        # Warmup, not recorded.
        for _ in range(WARMUPS):
            g=m['build_graph'](inst,delta=0.0,y_class=0)
            norm.apply(g)
            predict_one(model,g,task)
        trials=[]
        for _ in range(REPEATS):
            s=time.perf_counter();g=m['build_graph'](inst,delta=0.0,y_class=0);tg=time.perf_counter()-s
            s=time.perf_counter();norm.apply(g);tn=time.perf_counter()-s
            s=time.perf_counter();decision=predict_one(model,g,task);ti=time.perf_counter()-s
            trials.append((tg,tn,ti,decision))
        decisions={z[3] for z in trials}
        if len(decisions)!=1 or decision!=str(expected.loc[key]):
            raise RuntimeError(f'{kind} {task}: prediction mismatch for {key}, timed={decisions}, original={expected.loc[key]}; stop before combining times')
        vals=np.median(np.array([t[:3] for t in trials]),axis=0)
        results.append(dict(_key=key,experiment=kind,model='GNN-C' if task=='classification' else 'GNN-R',
                            graph_time_s=vals[0],normalize_time_s=vals[1],inference_time_s=vals[2],
                            gnn_overhead_s=float(sum(vals)),selected_relaxation=decision))
        if (idx+1)%25==0:print(kind,task,'timed',idx+1,'/',len(test),flush=True)
    result=pd.DataFrame(results)
    path=OUT/f'{kind}_{"gnn_c" if task=="classification" else "gnn_r"}_measured_times.csv'
    result.to_csv(path,index=False)
    print('Saved',path,'N=',len(result),'mean overhead=',result.gnn_overhead_s.mean())
    return result

rows=[]
for kind,m in [('root',root),('bb',bb)]:
    test,instances,norm=prepared[kind]
    for task in ['classification','regression']:
        rows.append(benchmark(kind,task,m,test,instances,norm))
times=pd.concat(rows,ignore_index=True)
times.to_csv(OUT/'gnn_measured_overhead_all.csv',index=False)
print(times.groupby(['experiment','model'])[['graph_time_s','normalize_time_s','inference_time_s','gnn_overhead_s']].agg(['mean','median']).round(6))

root classification timed 25 / 278
root classification timed 50 / 278
root classification timed 75 / 278
root classification timed 100 / 278
root classification timed 125 / 278
root classification timed 150 / 278
root classification timed 175 / 278
root classification timed 200 / 278
root classification timed 225 / 278
root classification timed 250 / 278
root classification timed 275 / 278
Saved C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\analysis3_gap_time_outputs_WITH_GNN\root_gnn_c_measured_times.csv N= 278 mean overhead= 0.826902999631012
root regression timed 25 / 278
root regression timed 50 / 278
root regression timed 75 / 278
root regression timed 100 / 278
root regression timed 125 / 278
root regression timed 150 / 278
root regression timed 175 / 278
root regression timed 200 / 278
root regression timed 225 / 278
root regression timed 250 / 278
root regression timed 275 / 278
Saved C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\analysis3_

## Integration with Analysis 3

In the existing Analysis 3, load `gnn_measured_overhead_all.csv`, match on **`Experiment` + `Strategy` + `_key`** (not row order), then add `gnn_overhead_s` to the previously stored per-instance `strategy_time_s` for GNN rows only. Do **not** add DI feature-extraction overhead to GNN. The normalized gap remains unchanged. Preserve 278 root and 280 B&B instances and recompute average times from per-instance rows.

**Note:** The file includes timing for actual QCQP graph preparation and prediction; model-load cost is intentionally excluded. Tree methods' sklearn inference cost has not been measured in Analysis 3 and should be disclosed if omitted.

In [4]:
# Integrate into Analysis 3's per-instance CSVs without recomputing LB/gap or training.
summary=[]
for kind in ['root','bb']:
    detail_path=OUT/f'{kind}_gap_time_per_instance.csv'
    if not detail_path.exists():
        print('SKIP missing',detail_path,'— first run Analysis 3');continue
    d=pd.read_csv(detail_path)
    assert {'_key','Strategy','strategy_time_s','solver_time_s','normalized_gap'}.issubset(d.columns)
    t=times.loc[times.experiment==kind,['_key','model','gnn_overhead_s']].rename(columns={'model':'Strategy'})
    assert not t.duplicated(['_key','Strategy']).any()
    x=d.merge(t,on=['_key','Strategy'],how='left',validate='many_to_one')
    is_gnn=x['Strategy'].isin(['GNN-C','GNN-R'])
    if x.loc[is_gnn,'gnn_overhead_s'].isna().any():
        raise RuntimeError(f'{kind}: missing measured GNN overhead for '+str(x.loc[is_gnn & x.gnn_overhead_s.isna(),'_key'].tolist()[:10]))
    x['gnn_overhead_s']=x['gnn_overhead_s'].fillna(0.0)
    x['strategy_time_s']=x['strategy_time_s']+x['gnn_overhead_s']
    out_path=OUT/f'{kind}_gap_time_per_instance_GNN_TIMED.csv';x.to_csv(out_path,index=False)
    s=x.groupby('Strategy',as_index=False).agg(N=('_key','size'),Avg_Normalized_Value=('normalized_gap','mean'),Avg_Time_s=('strategy_time_s','mean'),Avg_Solver_Time_s=('solver_time_s','mean'),Avg_GNN_Overhead_s=('gnn_overhead_s','mean'))
    s.insert(0,'Experiment',kind);summary.append(s)
    s.to_csv(OUT/f'{kind}_gap_time_points_GNN_TIMED.csv',index=False)
if summary:
    final=pd.concat(summary,ignore_index=True)
    final.to_csv(OUT/'analysis3_root_and_bb_gap_time_points_GNN_TIMED.csv',index=False)
    print(final.to_string(index=False))

Experiment    Strategy   N  Avg_Normalized_Value  Avg_Time_s  Avg_Solver_Time_s  Avg_GNN_Overhead_s
      root   Always LP 278              0.500422    0.104791           0.104791            0.000000
      root Always SDP' 278              0.366413    0.085342           0.085342            0.000000
      root Brute Force 278              0.319300    0.190133           0.190133            0.000000
      root         GBC 278              0.324119    9.018625           0.072838            0.000000
      root         GBR 278              0.355028    9.026646           0.080858            0.000000
      root       GNN-C 278              0.398098    0.915228           0.088325            0.826903
      root       GNN-R 278              0.383441    0.936243           0.086196            0.850047
      root       Ideal 278              0.319300    0.076862           0.076862            0.000000
      root         RFC 278              0.333897    9.014365           0.068577            0.000000
